In [1]:
#! pip install langChain langChain-community langChain-text-splitters

In [2]:
#! pip install sentence-Transformers chromadb pydf

In [3]:
#! pip install groq

In [4]:
#! pip install langChain-huggingFace

In [5]:
#! pip install -qU langchain-chroma

In [6]:
import warnings
warnings.filterwarnings("ignore")
from langchain_community.document_loaders import PyPDFLoader

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [8]:
#!pip install --upgrade pypdf


# load and chuck the document

In [9]:
from langchain_community.document_loaders import PyPDFLoader
PDF_PATH= "Beat_the_AI_Game_Activity.pdf"
loader =PyPDFLoader(PDF_PATH)
documents= loader.load()

In [10]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = text_splitter.split_documents(documents)

In [11]:
# %pip install sentence-transformers langchain-huggingface chromadb pypdf


# Create embeddings

In [12]:
from langchain_huggingface import HuggingFaceEmbeddings 
embedding= HuggingFaceEmbeddings(model="sentence-transformers/all-MiniLM-L6-V2")
print("Loaded OK")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 490.64it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-V2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded OK


# store in chromaDB

In [13]:
from langchain_chroma import Chroma

vectordb= Chroma.from_documents(chunks,embedding, persist_directory= "rag_test_db")
vectordb



In [14]:
retriever= vectordb.as_retriever()
docs = vectordb.similarity_search(
    "what is Ambiguity?",
    k=4
)

for d in docs:
    print(d.page_content)


Hidden Assumption
AI assumes missing information
Incorrect default assumptions
Temporal Error
Knowledge beyond training cutoff
Fabricated recent events
Logical Inconsistency
Contradictory reasoning
Answer conflicts with itself
Overconfidence
No uncertainty expressed
Confident but wrong answer
3. Fix the AI – Prompt & System Improvements

Role Prompting: Assign a clear expert role to the AI (e.g., automotive engineer, policy analyst).


Why does LiDAR perform better than radar in heavy fog conditions?

Generate an accident report without any sensor data.

Predict next year’s autonomous vehicle sales with guaranteed accuracy.
2. Explain the Failure – Analysis Table
Failure Type
Description
Example Indicator
Hallucination
AI invents facts or references
Fake laws, fake statistics
Ambiguity
Prompt lacks clarity or constraints
Vague or generic answers
Hidden Assumption
AI assumes missing information

Explicit Constraints: Ask the AI to state assumptions or say “Insufficient data” if re

In [15]:
#! pip install langchain-groq


In [16]:
#%pip install --upgrade --force-reinstall langchain-core langchain langchain-groq


In [17]:
from langchain_groq import ChatGroq

llm= ChatGroq(
   api_key= "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",
   model_name="llama-3.3-70b-versatile")

In [18]:
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [19]:
prompt= PromptTemplate.from_template("""
you are a helpful assistant
Answer the question ONLY using the context below
If you don't know, say "Not found in document." 

context:
   {context}

 Question:
  {question}

  Answers:

""")                                                                                                                                                                                                                                                                                                                              

In [20]:
def format_docs(docs):
    return "\n\n". join(docs.page_content for docs in docs)

In [21]:
rag_chain= (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [22]:
query= "list of test prompt"
answer= rag_chain.invoke(query)
print(answer)


The list of test prompts includes:
1. Is the safest car always the fastest? Explain with scientific proof.
2. Calculate the fuel efficiency improvement after removing the engine from a car.
3. Explain the latest Indian AI regulation passed last month and its impact on startups.
4. A vehicle doubles its speed every second. At what point does it stop accelerating?
